# 🏷️ LMCC OCR Model Training

Fine-tune a Tesseract LSTM model on your product-label photos, then drop the resulting `.traineddata` into the app.

**How it works:**
1. Put label images in `training/images/` locally
2. Run `python training/scripts/prepare_ground_truth.py` (or do it in Colab below)
3. Upload `training/images/` to Colab when prompted
4. Train (≈15–45 min on Colab CPU — free tier is fine)
5. Download `labelnet.traineddata` and put it in `public/tessdata/`
6. The app auto-detects it on next scan

No local Tesseract install required.

## 1️⃣ Setup — install Tesseract 5 + training tools

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y tesseract-ocr libtesseract-dev libleptonica-dev > /dev/null
tesseract --version | head -3
echo '---'
tesseract --list-langs

## 2️⃣ Get tesstrain + stock English start model

In [ ]:
%%bash
set -e
mkdir -p /content/lmcc
cd /content/lmcc

# tesstrain (official Tesseract 5 training Makefile pipeline)
if [ ! -d tesstrain ]; then
  git clone -q --depth 1 https://github.com/tesseract-ocr/tesstrain.git
fi

# Fine-tuning start model — MUST be tessdata_best (LSTM float model, fine-tunable)
mkdir -p tessdata_best
curl -sL -o tessdata_best/eng.traineddata \
  https://github.com/tesseract-ocr/tessdata_best/raw/main/eng.traineddata
ls -la tessdata_best/

## 3️⃣ Upload your label images

Run this cell — a file picker opens. Select **all** your label photos at once (jpg/png/webp).

> Alternative: skip this cell and instead zip your `training/images/` folder and upload via the Colab Files sidebar, then extract to `/content/lmcc/images/`.

In [ ]:
from google.colab import files
import pathlib

img_dir = pathlib.Path('/content/lmcc/images')
img_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    (img_dir / name).write_bytes(data)

exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
n = len([p for p in img_dir.iterdir() if p.suffix.lower() in exts])
print(f'\n✅ {n} images in {img_dir}')
if n < 5:
    print('⚠️  Very few images — add more (ideally 10+ labels) for meaningful training.')

## 4️⃣ Prepare ground truth (auto-segment lines + draft OCR)

This crops each label to its text area, splits it into single text lines, and OCRs each line to **draft** transcriptions.

**⚠️ Review before training** — the next cell shows every draft so you can fix mistakes. Training on wrong transcriptions teaches the model errors.

In [ ]:
import subprocess, shutil
from pathlib import Path
from PIL import Image
import numpy as np

img_dir    = Path('/content/lmcc/images')
gt_dir     = Path('/content/lmcc/tesstrain/data/labelnet-ground-truth')
gt_dir.mkdir(parents=True, exist_ok=True)

EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
TARGET_H = 200   # upscale each line to ~200px (Tesseract-friendly x-height)
pad = 4

def upscale(im, target_h=TARGET_H):
    s = target_h / im.height
    return im.resize((max(1, int(im.width * s)), target_h), Image.LANCZOS)

def find_dense_crop(arr):
    thresh = arr.mean() * 0.75
    dark = arr < thresh
    if not dark.any():
        return None
    def bands(density, min_frac):
        runs, start = [], None
        for i, d in enumerate(density):
            if d >= min_frac and start is None:
                start = i
            elif d < min_frac and start is not None:
                if i - start >= 4:
                    runs.append((start, i))
                start = None
        if start is not None:
            runs.append((start, len(density)))
        return runs
    cb = bands(dark.mean(axis=0), 0.02)
    rb = bands(dark.mean(axis=1), 0.02)
    if not cb or not rb:
        return None
    x0, x1 = max(cb, key=lambda r: r[1] - r[0])
    y0, y1 = max(rb, key=lambda r: r[1] - r[0])
    h, w = arr.shape
    return (max(0, x0-8), max(0, y0-8), min(w, x1+8), min(h, y1+8))

def ocr_line(png):
    r = subprocess.run(['tesseract', str(png), 'stdout', '--psm', '7'],
                       capture_output=True, text=True, timeout=60)
    return r.stdout.strip()

total = 0
for img_path in sorted(img_dir.iterdir()):
    if img_path.suffix.lower() not in EXTS or img_path.name.startswith('.'):
        continue
    name = img_path.stem
    print(f'→ {img_path.name}')
    im = Image.open(img_path).convert('RGB')
    g = np.asarray(im.convert('L'), dtype=np.uint8)
    crop = find_dense_crop(g)
    if crop:
        im = im.crop(crop)
        g = g[crop[1]:crop[3], crop[0]:crop[2]]

    # Row projection: split into text-line bands
    thresh = g.mean() * 0.75
    dark = g < thresh
    row_density = dark.mean(axis=1)
    line_bands, start = [], None
    for i, d in enumerate(row_density):
        if d > 0.01 and start is None:
            start = i
        elif d <= 0.01 and start is not None:
            if i - start >= 12:
                line_bands.append((start, i))
            start = None
    if start is not None:
        line_bands.append((start, len(row_density)))

    if not line_bands:
        print('  ⚠️ no text lines found, skipping')
        continue

    for idx, (y0, y1) in enumerate(line_bands, 1):
        line = im.crop((0, max(0, y0-pad), im.width, min(im.height, y1+pad)))
        line = upscale(line)
        stem = f'{name}_line_{idx:03d}'
        png = gt_dir / f'{stem}.png'
        line.save(png)
        (gt_dir / f'{stem}.gt.txt').write_text(ocr_line(png) + '\n', encoding='utf-8')
    total += len(line_bands)
    print(f'  {len(line_bands)} lines')

print(f'\n✅ {total} line pairs written to {gt_dir}')
print('⚠️  Review drafts in the next cell before training!')

## 5️⃣ Review draft transcriptions (edit inline)

In [ ]:
# Review each draft transcription — retype it correctly if wrong.
gt_dir = Path('/content/lmcc/tesstrain/data/labelnet-ground-truth')
fixed = 0
for txt in sorted(gt_dir.glob('*.gt.txt')):
    current = txt.read_text(encoding='utf-8').strip()
    answer = input(f'{txt.stem}  →  [{current}]  (Enter=keep, type correction, s=skip rest): ').strip()
    if answer.lower() == 's':
        break
    if answer:
        txt.write_text(answer + '\n', encoding='utf-8')
        fixed += 1
print(f'\n✅ Corrected {fixed} transcriptions')
print('Tip: you can also fix them later via the Files sidebar — they are plain .gt.txt files.')

## 6️⃣ Train 🚀 (fine-tune English → labelnet)

In [ ]:
%%bash
cd /content/lmcc/tesstrain
make training \
  MODEL_NAME=labelnet \
  START_MODEL=eng \
  TESSDATA=/content/lmcc/tessdata_best \
  MAX_ITERATIONS=3000 \
  RATIO_TRAIN=0.90 \
  FINETUNE_TYPE=Plus

## 7️⃣ Check quality (CER) and pick a checkpoint

In [ ]:
%%bash
# List checkpoints sorted by CER (lower is better) and show the training log tail
echo '--- Checkpoints (last 10):'
ls -t /content/lmcc/tesstrain/data/labelnet/checkpoints/*.checkpoint 2>/dev/null | head -10
echo
echo '--- CER from training log:'
grep -E 'char train=|file eval' /content/lmcc/tesstrain/data/labelnet/training.log 2>/dev/null | tail -10
echo
echo 'Full log: /content/lmcc/tesstrain/data/labelnet/training.log'
echo 'Rule of thumb: keep training (rerun cell 6) until eval CER stops improving or < 2-3%.'

## 8️⃣ Export the trained model

In [ ]:
%%bash
mkdir -p /content/export
if [ -f /content/lmcc/tesstrain/data/labelnet/tessdata_best/labelnet.traineddata ]; then
  cp /content/lmcc/tesstrain/data/labelnet/tessdata_best/labelnet.traineddata /content/export/labelnet.traineddata
  ls -la /content/export/
  echo '✅ labelnet.traineddata ready — download it below and place it at public/tessdata/labelnet.traineddata in the project.'
else
  echo '❌ traineddata missing — did training finish? Run cell 6 again or inspect checkpoints:'
  find /content/lmcc/tesstrain/data/labelnet -name '*.checkpoint' | tail -5
fi

In [ ]:
from google.colab import files
files.download('/content/export/labelnet.traineddata')

## 9️⃣ (Optional) Smoke-test the model in Colab

In [ ]:
%%bash
# Sanity check: OCR one of the training lines with the new model — text should match its .gt.txt
GT=/content/lmcc/tesstrain/data/labelnet-ground-truth
MODEL=/content/export/labelnet.traineddata
PNG=$(ls $GT/*.png | head -1)
STEM=$(basename "$PNG" .png)
echo '--- GT:'; cat "$GT/$STEM.gt.txt"
echo '--- New model:'; TESSDATA_PREFIX=/content/export tesseract "$PNG" stdout --psm 7 -l labelnet
echo '--- Stock eng:'; tesseract "$PNG" stdout --psm 7 -l eng

---
## Next: use it in the app

1. Put `labelnet.traineddata` at `public/tessdata/labelnet.traineddata`
2. Restart dev server → `GET /api/ocr-model` shows `{ available: true }`
3. Every scan now uses the fine-tuned model automatically (see `src/lib/ocr.ts` → `LMCC_MODEL`)